# 👗 Garment Image Beautifier (nano banana)

Beautifies garment photos in bulk with Google's **Gemini 2.5 Flash Image** ("nano banana").

This version reads your images straight from **GitHub** — no Google Drive, no pop-ups.

**How to use it:**
1. Run each cell in order (click the ▶️ on the left, top to bottom).
2. Paste your Gemini API key when Step 2 asks.
3. Step 4 beautifies your images and downloads a **zip** of the results to your computer.

To add or change images, upload them to the `input/` folder of your GitHub repo, then re-run.

## Step 1 — Install the tools
Run once. ~20 seconds.

In [ ]:
!pip install -q google-genai pillow requests
print('✅ Done installing.')

## Step 2 — Paste your Gemini API key
Get one at https://aistudio.google.com/apikey. A box appears when you run this — paste your key there (it stays hidden).

In [ ]:
from getpass import getpass
API_KEY = getpass('Paste your Gemini API key and press Enter: ')
print('✅ Key saved for this session.')

## Step 3 — Your prompt & settings
Your prompt is already loaded. Change the repo only if you use a different one.

In [ ]:
# 👇 YOUR PROMPT (loaded and ready — you can edit it here anytime)
PROMPT = """
A professional e-commerce product photograph of the garment shown in the provided image, presented in a 1:1 square aspect ratio. The item is positioned perfectly dead-center with a generous, uniform margin of approximately 10% to 15% negative space on the top, bottom, left, and right sides, ensuring the entire garment is safely contained within the frame and no edges or corners are cut off. The item is presented in a clean, perfectly symmetrical and de-wrinkled true flat lay, where the garment is spread completely flat on the surface. The background is a seamless, solid light gray studio backdrop with a specific hex code of #D7D7D7. A subtle, soft drop shadow is cast directly behind the item with no horizontal offset (x=0) and no vertical offset (y=0), giving the item a clean, distinct lift from the surface.

Crucially, all authentic brand details and logos must be preserved as pristine, identical elements; there must be zero hallucination, addition, or alteration of any logos, text, or symbols.

Studio lighting is bright, even, and diffused, provided by large softboxes to eliminate harsh reflections and provide clean details of the fabric structure and all intrinsic details. The focus is sharp across the entire garment, capturing intricate texture without being washed out. The presentation is immaculately clean and premium.
"""

# Where your images live on GitHub
GITHUB_REPO = 'michellefleek/garment-images'   # owner/repo
GITHUB_BRANCH = 'main'
INPUT_PATH = 'input'                            # folder in the repo with your images

# The nano banana model
MODEL = 'gemini-2.5-flash-image'

# 🧪 TEST MODE: only process this many images (great for a first run).
# Set to None to process ALL images in the input folder.
TEST_LIMIT = 5

print('✅ Settings loaded.')

## Step 4 — Run the batch
Reads each image from GitHub, beautifies it, and (when finished) downloads a zip of the results.

Safe to stop and re-run — already-finished images are skipped.

In [ ]:
import os, time, io, requests
from PIL import Image
from google import genai
from google.colab import files

client = genai.Client(api_key=API_KEY)
OUTPUT_DIR = '/content/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
IMAGE_EXTS = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')
MAX_RETRIES = 4

def list_github_images():
    url = f'https://api.github.com/repos/{GITHUB_REPO}/contents/{INPUT_PATH}?ref={GITHUB_BRANCH}'
    r = requests.get(url, headers={'Accept': 'application/vnd.github+json'})
    r.raise_for_status()
    items = r.json()
    return [it for it in items if it['type'] == 'file' and it['name'].lower().endswith(IMAGE_EXTS)]

def beautify(img_bytes, out_path):
    img = Image.open(io.BytesIO(img_bytes))
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.models.generate_content(model=MODEL, contents=[PROMPT, img])
            for part in resp.candidates[0].content.parts:
                if getattr(part, 'inline_data', None) and part.inline_data.data:
                    Image.open(io.BytesIO(part.inline_data.data)).save(out_path)
                    return True
            return False  # no image returned (e.g. safety block)
        except Exception as e:
            wait = min(2 ** attempt, 30)
            print(f'   ⚠️  attempt {attempt} failed ({e}); retrying in {wait}s...')
            time.sleep(wait)
    return False

images = list_github_images()
if TEST_LIMIT:
    images = images[:TEST_LIMIT]
    print(f'🧪 TEST MODE: only processing the first {TEST_LIMIT} image(s).')
print(f'Found {len(images)} image(s) to process.\n')

done = skipped = failed = 0
for i, it in enumerate(images, 1):
    name = it['name']
    out_path = os.path.join(OUTPUT_DIR, os.path.splitext(name)[0] + '.png')
    if os.path.exists(out_path):
        skipped += 1
        print(f'[{i}/{len(images)}] ⏭️  {name} (already done)')
        continue
    print(f'[{i}/{len(images)}] 🎨 {name} ...')
    img_bytes = requests.get(it['download_url']).content
    if beautify(img_bytes, out_path):
        done += 1
        print('          ✅ saved')
    else:
        failed += 1
        print('          ❌ could not beautify this one (skipping)')
    time.sleep(1)  # gentle pacing

print(f'\n🏁 Finished. {done} new, {skipped} already done, {failed} failed.')

# Zip up the results and download them to your computer
if done or skipped:
    import shutil
    shutil.make_archive('/content/beautified', 'zip', OUTPUT_DIR)
    print('⬇️  Downloading beautified.zip to your computer...')
    files.download('/content/beautified.zip')